# Ejecutar tu parte en Databricks

Abre desde la Git folder. Este notebook llama módulos del repositorio. No modifica tablas GDELT existentes. Primer paso: probar precios; noticias solo con acceso válido.

In [ ]:
%pip install -r ../requirements.txt

Si Databricks solicita reiniciar Python, hazlo antes de las siguientes celdas.

In [ ]:
from pathlib import Path
import sys, os
project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p/'config/config.example.json').exists()), None)
if project_root is None:
    raise RuntimeError('Abre este notebook desde notebooks/ dentro del repositorio.')
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
data_root = Path(os.getenv('STOCK_DATA_ROOT', str(project_root/'data')))
print('Raíz del proyecto:', project_root)


In [ ]:
dbutils.widgets.text('data_root', '', 'Subcarpeta de un Volume existente')
volume_path = dbutils.widgets.get('data_root').strip()
if not volume_path.startswith('/Volumes/') or len(Path(volume_path).parts) < 6:
    raise RuntimeError('Escribe /Volumes/CATALOGO/ESQUEMA/VOLUME/stock_pipeline en el widget data_root y vuelve a correr esta celda.')
data_root = Path(volume_path)
data_root.mkdir(parents=True, exist_ok=True)
os.environ['STOCK_DATA_ROOT'] = str(data_root)
print('Ubicación de datos configurada.')


Configura un Volume existente desde Catalog. Ejemplo de estructura, no ruta real: /Volumes/catalogo/esquema/volume/stock_pipeline. No pegues una API key en un widget. Si no tienes Volume, crea uno desde Catalog con permisos propios o ejecuta los scripts localmente.

In [ ]:
from src.utils.common import config, write_json, fingerprint
from src.ingestion.download_data import download_prices, download_news
cfg = config(project_root/'config/config.example.json')
price_path, metadata = download_prices(cfg, data_root)
write_json(data_root/'raw/prices_manifest.json', {'config':fingerprint(cfg), 'file':str(price_path.relative_to(data_root)), 'sha256':metadata['sha256']})
print('Precios:', metadata['rows'])


Noticias: NEWS_SENTIMENT está documentado como premium. Comprueba tu acceso antes de correr. No subir clave ni respuestas completas al repositorio. Si internet del workspace bloquea el proveedor, descarga con el script local y copia raw/ íntegro, incluyendo manifiestos, al mismo Volume.

In [ ]:
from getpass import getpass
if not os.getenv('ALPHAVANTAGE_API_KEY'):
    os.environ['ALPHAVANTAGE_API_KEY'] = getpass('Alpha Vantage API key: ')
try:
    download_news(cfg, data_root)
finally:
    os.environ.pop('ALPHAVANTAGE_API_KEY', None)


In [ ]:
from src.processing.build_dataset import build
df = build(cfg, data_root)
display(df.head())


In [ ]:
from src.processing.explore_data import explore
report_root = data_root/'reports'
summary = explore(data_root, report_root)
print('Cobertura de sesiones con noticias:', summary['news_coverage'])


Ahora abre 01_exploration.ipynb. Registra evidencia real y sube solo las muestras autorizadas/documentación. Validación independiente, tests y benchmark siguen pendientes del equipo.